# ISLES'26 — End-to-End Colab Walkthrough

This notebook covers the full pipeline on **synthetic data** (no challenge registration required):

1. Install dependencies
2. Clone the repository
3. Generate synthetic T1w MRI + stroke lesion masks
4. Run preprocessing
5. Create 3-fold splits
6. Train SegResNet+FiLM (fast debug run, 2 epochs)
7. Evaluate with stratified metrics
8. Ensemble inference on a new volume
9. Visualise predictions

> **Runtime**: GPU (T4 recommended). CPU-only will work for the debug run but is ~10× slower.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hashirama21/isles2026-Ischemic-Stroke-Lesion-Segmentation/blob/main/notebooks/02_colab_e2e.ipynb)

## 1 · Install dependencies

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --depth 1 https://github.com/hashirama21/isles2026-Ischemic-Stroke-Lesion-Segmentation.git /content/isles26
    %cd /content/isles26
    !pip install -q -e '.[dev]'
    print('Installed.')
else:
    import os
    os.chdir('/Users/krohn/PycharmProjects/isles26')  # local dev path
    print('Local mode.')

In [ ]:
import torch
import pytorch_lightning as pl
import monai

print(f'PyTorch        {torch.__version__}')
print(f'Lightning      {pl.__version__}')
print(f'MONAI          {monai.__version__}')
print(f'CUDA available {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            {torch.cuda.get_device_name(0)}')

## 2 · Generate synthetic ISLES'26-style data

Each synthetic subject has:
- a 96³ T1w brain volume (Gaussian random fields + skull mask)
- a small ellipsoidal lesion mask
- metadata JSON with `DAYS_POST_STROKE`, `CHRONICITY`, `CENTER`

In [ ]:
import json
import random
from pathlib import Path

import nibabel as nib
import numpy as np

random.seed(42)
np.random.seed(42)

RAW_DIR = Path('data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

CENTERS = ['CTR_A', 'CTR_B', 'CTR_C', 'CTR_D']
N_SUBJECTS = 24
VOL_SHAPE = (96, 96, 96)
AFFINE = np.diag([1.5, 1.5, 1.5, 1.0])


def _make_ellipsoid(shape, center, radii):
    """Binary ellipsoid mask inside a volume."""
    D, H, W = shape
    z, y, x = np.ogrid[:D, :H, :W]
    cz, cy, cx = center
    rz, ry, rx = radii
    return ((z - cz)**2 / rz**2 + (y - cy)**2 / ry**2 + (x - cx)**2 / rx**2) <= 1


def _make_brain(shape):
    """Rough brain-shaped mask (central sphere + noise)."""
    c = [s // 2 for s in shape]
    mask = _make_ellipsoid(shape, c, [s * 0.45 for s in shape])
    return mask


for i in range(N_SUBJECTS):
    subj_id = f'sub-{i+1:03d}'
    subj_dir = RAW_DIR / subj_id
    subj_dir.mkdir(exist_ok=True)

    brain_mask = _make_brain(VOL_SHAPE)

    # T1w: Gaussian noise, bright inside brain
    t1w = np.random.randn(*VOL_SHAPE).astype(np.float32)
    t1w[brain_mask] += 3.0
    t1w[~brain_mask] = 0.0

    # Lesion: small ellipsoid in a random hemisphere
    has_lesion = random.random() > 0.25  # 75 % have lesion
    label = np.zeros(VOL_SHAPE, dtype=np.uint8)
    if has_lesion:
        cx = random.randint(30, 65)
        cy = random.randint(30, 65)
        cz = random.randint(30, 65)
        rx = random.randint(3, 8)
        ry = random.randint(3, 8)
        rz = random.randint(3, 8)
        ell = _make_ellipsoid(VOL_SHAPE, (cz, cy, cx), (rz, ry, rx))
        label[ell & brain_mask] = 1

    nib.save(nib.Nifti1Image(t1w, AFFINE), subj_dir / f'{subj_id}_T1w.nii.gz')
    nib.save(nib.Nifti1Image(label, AFFINE), subj_dir / f'{subj_id}_mask.nii.gz')

    days = random.randint(0, 365)
    chronicity = 0 if days < 7 else (1 if days < 90 else 2)
    meta = {
        'DAYS_POST_STROKE': days,
        'CHRONICITY': chronicity,
        'CENTER': random.choice(CENTERS),
    }
    with open(subj_dir / f'{subj_id}_meta.json', 'w') as f:
        json.dump(meta, f)

print(f'Generated {N_SUBJECTS} synthetic subjects in {RAW_DIR}/')

## 3 · Preprocessing

Skull-strip (intensity fallback when HD-BET is unavailable) + Z-score normalisation.

In [ ]:
!python scripts/preprocess.py \
    data.raw_dir=data/raw \
    data.out_dir=data/processed \
    preprocess.n_jobs=2

In [ ]:
processed = sorted(Path('data/processed').glob('*/metadata.json'))
print(f'{len(processed)} processed subjects')
with open(processed[0]) as f:
    import json; print(json.dumps(json.load(f), indent=2))

## 4 · Create cross-validation splits

In [ ]:
!python scripts/make_splits.py splits.n_folds=3 splits.test_ratio=0.15

In [ ]:
with open('data/splits/splits_5fold.json') as f:
    splits = json.load(f)

for fold, data in splits.items():
    print(f"{fold}: train={len(data['train'])}  val={len(data['val'])}  test={len(data['test'])}")

## 5 · Training (debug run — 2 epochs, SegResNet+FiLM)

We use the `segresnet_baseline` experiment (no deep supervision) and `debug=true` (2 epochs, fast_dev_run) to verify the full training loop without W&B logging.

In [ ]:
!python scripts/train.py \
    experiment=segresnet_baseline \
    fold=0 \
    debug=true \
    training.max_epochs=2 \
    data.num_workers=0 \
    wandb.mode=disabled

## 6 · Full training run (optional — set `RUN_FULL_TRAIN = True`)

Train for 50 epochs on fold 0. Skip in demo mode.

In [ ]:
RUN_FULL_TRAIN = False

if RUN_FULL_TRAIN:
    !python scripts/train.py \
        experiment=segresnet_baseline \
        fold=0 \
        training.max_epochs=50 \
        data.num_workers=2 \
        wandb.mode=disabled
else:
    print('Skipped full training. Set RUN_FULL_TRAIN=True to enable.')

## 7 · Evaluation with stratified metrics

Find the best checkpoint from fold 0 and run the evaluation script.

In [ ]:
import glob

ckpts = sorted(glob.glob('outputs/checkpoints/fold0/*.ckpt'))
if not ckpts:
    print('No checkpoint found — skipping evaluation (debug run does not save checkpoints).')
    CKPT = None
else:
    CKPT = ckpts[-1]
    print(f'Using checkpoint: {CKPT}')

In [ ]:
if CKPT:
    !python scripts/evaluate.py \
        checkpoint={CKPT} \
        fold=0 \
        data.num_workers=0 \
        experiment=segresnet_baseline
else:
    print('Skipped — no checkpoint available.')

## 8 · Direct inference on a single volume

Demonstrate the full predict pipeline without going through the training script.

In [ ]:
import numpy as np
import torch
import nibabel as nib
from omegaconf import OmegaConf
from pathlib import Path

from src.inference.predict import ensemble_predict

# Load the first test subject
with open(processed[0]) as f:
    meta = json.load(f)

img_nib = nib.load(meta['image_path'])
data = np.asarray(img_nib.dataobj, dtype=np.float32)

image_tensor = torch.from_numpy(data).unsqueeze(0).unsqueeze(0)  # [1,1,D,H,W]
meta_tensor = torch.tensor(
    [min(float(meta.get('days_post_stroke', 0)) / 365.0, 1.0),
     float(meta.get('chronicity', 0))],
    dtype=torch.float32,
).unsqueeze(0)  # [1,2]

print(f'Image shape : {image_tensor.shape}')
print(f'Meta tensor : {meta_tensor}')

In [ ]:
postprocess_cfg = OmegaConf.create({
    'postprocess': {
        'min_lesion_volume_ml': 0.05,
        'threshold_acute': 0.45,
        'threshold_chronic': 0.35,
    }
})

model_cfg = OmegaConf.create({
    '_target_': 'src.models.segresnet.SegResNetFiLM',
    'in_channels': 1,
    'out_channels': 2,
    'init_filters': 32,
    'blocks_down': [1, 2, 2, 4],
    'blocks_up': [1, 1, 1],
    'dropout_prob': 0.2,
    'meta_dim': 2,
    'film_hidden_dim': 64,
})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if CKPT:
    pred = ensemble_predict(
        checkpoint_paths=[Path(CKPT)],
        model_cfgs=[model_cfg],
        image=image_tensor,
        metadata=meta_tensor,
        roi_size=[64, 64, 64],
        use_tta=False,
        device=device,
        postprocess_cfg=postprocess_cfg,
        voxel_spacing_mm=tuple(float(z) for z in img_nib.header.get_zooms()[:3]),
    )
    print(f'Prediction shape : {pred.shape}  unique values: {np.unique(pred)}')
else:
    print('No checkpoint — generating dummy prediction for visualization demo.')
    pred = np.zeros(data.shape, dtype=np.uint8)
    pred[40:50, 40:50, 40:50] = 1

## 9 · Visualisation

Show 3 axial slices: T1w image, ground-truth lesion mask, predicted lesion mask.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

gt_nib = nib.load(meta['label_path'])
gt = np.asarray(gt_nib.dataobj, dtype=np.uint8)

D = data.shape[0]
slices = [D // 4, D // 2, 3 * D // 4]

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for col, sl in enumerate(slices):
    t1_slice = data[sl]
    gt_slice = gt[sl]
    pred_slice = pred[sl]

    axes[0, col].imshow(t1_slice, cmap='gray')
    axes[0, col].set_title(f'T1w — slice {sl}')
    axes[0, col].axis('off')

    axes[1, col].imshow(t1_slice, cmap='gray')
    if gt_slice.any():
        axes[1, col].imshow(np.ma.masked_where(gt_slice == 0, gt_slice),
                            cmap='Reds', alpha=0.6, vmin=0, vmax=1)
    axes[1, col].set_title(f'Ground truth — slice {sl}')
    axes[1, col].axis('off')

    axes[2, col].imshow(t1_slice, cmap='gray')
    if pred_slice.any():
        axes[2, col].imshow(np.ma.masked_where(pred_slice == 0, pred_slice),
                            cmap='Blues', alpha=0.6, vmin=0, vmax=1)
    axes[2, col].set_title(f'Prediction — slice {sl}')
    axes[2, col].axis('off')

red_patch = mpatches.Patch(color='red', alpha=0.6, label='Ground truth')
blue_patch = mpatches.Patch(color='blue', alpha=0.6, label='Prediction')
fig.legend(handles=[red_patch, blue_patch], loc='lower center', ncol=2, fontsize=12)
fig.suptitle(f"Subject: {meta['subject_id']}  |  Days post stroke: {meta.get('days_post_stroke', '?')}",
             fontsize=14)
plt.tight_layout()
plt.show()

## 10 · Per-lesion metrics on the test subject

In [ ]:
from src.evaluation.metrics import compute_metrics

spacing = tuple(float(z) for z in img_nib.header.get_zooms()[:3])
metrics = compute_metrics(pred, gt, voxel_spacing_mm=spacing)

print('Dice score        :', round(metrics['dice'], 4))
print('Precision         :', round(metrics['precision'], 4))
print('Recall            :', round(metrics['recall'], 4))
if metrics.get('hd95') is not None:
    print('HD95 (mm)         :', round(metrics['hd95'], 2))
    print('ASSD (mm)         :', round(metrics['assd'], 2))
print('Pred volume (mL)  :', round(metrics['pred_volume_ml'], 4))
print('GT   volume (mL)  :', round(metrics['gt_volume_ml'], 4))

## 11 · Postprocessing walkthrough

Inspect the effect of each postprocessing step individually.

In [ ]:
import numpy as np
from src.inference.postprocess import remove_small_components, fill_holes, adaptive_threshold

# Build a fake probability map for demonstration
prob_map = np.zeros(data.shape, dtype=np.float32)
prob_map[40:55, 40:55, 40:55] = 0.85    # main lesion
prob_map[70:72, 70:72, 70:72] = 0.55    # small spurious blob

# 1. Threshold
days = float(meta.get('days_post_stroke', 0))
binary = adaptive_threshold(prob_map, days_post_stroke=days)
print(f'After threshold    : {binary.sum()} voxels')

# 2. Remove small components
vox_vol_ml = float(np.prod(spacing)) * 0.001
min_vox = 0.05 / vox_vol_ml
cleaned = remove_small_components(binary, min_volume_vox=min_vox)
print(f'After CC filtering : {cleaned.sum()} voxels')

# 3. Hole filling
filled = fill_holes(cleaned)
print(f'After hole fill    : {filled.sum()} voxels')

## 12 · Weighted lesion sampler

Show how the sampler weights are distributed across the dataset.

In [ ]:
from src.data.sampler import build_lesion_sampler
import matplotlib.pyplot as plt

samples = [json.loads(p.read_text()) for p in processed]

sampler = build_lesion_sampler(samples, small_lesion_threshold_ml=1.0)
weights = sampler.weights.numpy()

from collections import Counter
wt_counts = Counter(weights)
print('Weight distribution:')
for w, cnt in sorted(wt_counts.items()):
    label = {0.5: 'no lesion', 1.0: 'large lesion', 3.0: 'small lesion'}.get(w, str(w))
    print(f'  {w:.1f} ({label}): {cnt} subjects')

plt.figure(figsize=(5, 3))
plt.bar([str(k) for k in sorted(wt_counts)], [wt_counts[k] for k in sorted(wt_counts)],
        color=['#d62728', '#1f77b4', '#2ca02c'])
plt.xlabel('Sampling weight')
plt.ylabel('# subjects')
plt.title('Lesion sampler weight distribution')
plt.tight_layout()
plt.show()

## 13 · FiLM conditioning sanity check

Verify that different metadata values produce different bottleneck activations.

In [ ]:
from hydra import initialize_config_dir, compose
from hydra.utils import instantiate
import os

config_dir = str(Path(os.getcwd()) / 'configs')

with initialize_config_dir(config_dir=config_dir, version_base=None):
    cfg = compose(config_name='config', overrides=['experiment=segresnet_baseline'])

model = instantiate(cfg.model).eval()

dummy_img = torch.zeros(1, 1, 64, 64, 64)
meta_acute   = torch.tensor([[0.01, 0.0]])   # 3.65 days, acute
meta_chronic = torch.tensor([[0.80, 2.0]])   # ~292 days, chronic

with torch.no_grad():
    out_acute   = model(dummy_img, meta_acute)
    out_chronic = model(dummy_img, meta_chronic)

diff = (out_acute - out_chronic).abs().mean().item()
print(f'Output diff (acute vs chronic): {diff:.6f}')
print('FiLM conditioning is active.' if diff > 1e-6 else 'WARNING: FiLM has no effect!')

## 14 · Next steps

| Step | Command |
|------|---------|
| Full 5-fold training | `python scripts/train.py fold={0..4} training.max_epochs=500` |
| Evaluate each fold | `python scripts/evaluate.py checkpoint=outputs/checkpoints/fold{N}/best.ckpt fold={N}` |
| Ensemble inference | `python scripts/ensemble.py ensemble.use_tta=true` |
| Build Docker image | `bash docker/build.sh` |
| Run tests | `pytest tests/ -v` |

For the full Grand Challenge submission, replace synthetic data with the official ISLES'26 dataset available from [isles-challenge.org](https://www.isles-challenge.org).